In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# Create pilot subset (1260 images)

# Paths
BASE_DIR       = Path("..")
HAM_IMG_DIR    = BASE_DIR / "data" / "ham10000" / "images"
ISIC_IMG_DIR   = BASE_DIR / "data" / "isic2018" / "images"
HAM_CSV        = BASE_DIR / "data" / "preprocessed_manifests" / "ham10000_preprocessed.csv"
ISIC_CSV       = BASE_DIR / "data" / "preprocessed_manifests" / "isic2018_preprocessed.csv"
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Updated directory name to reflect new size
PILOT_1260_DIR = BASE_DIR / "data" / "pilot_1260"
PILOT_1260_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

# Updated targets (630 each = 1260 total)
PILOT_SIZE_HAM = 630
PILOT_SIZE_ISIC = 630

# ------------------------------------------------------------
# Load manifests

ham_df = pd.read_csv(HAM_CSV)
isic_df = pd.read_csv(ISIC_CSV)

ham_df["dataset"] = "HAM10000"
isic_df["dataset"] = "ISIC2018"

# ------------------------------------------------------------
# HAM sampling (Scaled to 315 each)

ham_mel = ham_df[ham_df["label"] == 1]
ham_nonmel = ham_df[ham_df["label"] == 0]

ham_sample = pd.concat([
    ham_mel.sample(
        n=315,
        random_state=RANDOM_STATE
    ),
    ham_nonmel.sample(
        n=315,
        random_state=RANDOM_STATE
    )
])

# ------------------------------------------------------------
# ISIC sampling (Scaled proportionally)
# ------------------------------------------------------------

very_tiny = isic_df[isic_df["mask_size_class"] == "very_tiny"]
tiny = isic_df[isic_df["mask_size_class"] == "tiny"]
small = isic_df[isic_df["mask_size_class"] == "small"]
normal = isic_df[isic_df["mask_size_class"] == "normal"]
border_touching = isic_df[isic_df["touches_any_border"] == True]

isic_sample = pd.concat([
    very_tiny.sample(n=63, random_state=RANDOM_STATE),
    tiny.sample(n=63, random_state=RANDOM_STATE),
    small.sample(n=63, random_state=RANDOM_STATE),
    normal.sample(n=378, random_state=RANDOM_STATE),
    border_touching.sample(n=63, random_state=RANDOM_STATE), # may overlap with the above categories
])

# ------------------------------------------------------------
# Merge

pilot_df = pd.concat([
    ham_sample,
    isic_sample
]).reset_index(drop=True)

# ------------------------------------------------------------
# Save

pilot_csv = PILOT_1260_DIR / "pilot_subset_1260.csv"
pilot_df.to_csv(pilot_csv, index=False)

print("Saved:", pilot_csv)
print("Total images:", len(pilot_df))

display(
    pilot_df.groupby("dataset").size()
)

Because we are grabbing a significantly larger slice of the ISIC dataset now, we should keep an eye on your border_touching category.

border_touching can overlap with the size categories (very_tiny, tiny, etc.). Because pd.concat just stacks DataFrames blindly, a row that is both normal and border_touching might get picked twice, creating a duplicate row in your final pilot_df.

If our total final count outputs slightly less than 1,260 after a .drop_duplicates() step later on, or if we notice identical image IDs in isic_sample, this overlap is why!